<a href="https://colab.research.google.com/github/purnimakushwaha/ITC101_Minor-project_python/blob/main/GitHub_profile_explorer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
#                 GITHUB PROFILE EXPLORER
# ============================================================

import requests
import pandas as pd
import os
import time
import webbrowser
from datetime import datetime


# ============================================================
# API SETTINGS
# ============================================================

BASE_URL = "https://api.github.com"

FAVORITES_FILE = "github_favorites.csv"
HISTORY_FILE = "github_history.csv"


# ============================================================
# PROJECT DATA
# ============================================================

current_profile = None

current_repositories = []

favorites = []

history = []


# ============================================================
# API REQUEST FUNCTION
# ============================================================

def api_request(url):

    try:

        start_time = time.perf_counter()

        response = requests.get(
            url,
            timeout=15
        )

        end_time = time.perf_counter()

        response_time = end_time - start_time

        print(
            f"\nAPI Response Time: "
            f"{response_time:.4f} seconds"
        )


        if response.status_code == 200:

            return response.json()


        elif response.status_code == 404:

            print(
                "\n❌ User or data not found."
            )

            return None


        elif response.status_code == 403:

            print(
                "\n⚠️ GitHub API rate limit reached."
            )

            return None


        else:

            print(
                "\n❌ API Error"
            )

            print(
                "Status Code:",
                response.status_code
            )

            return None


    except requests.exceptions.Timeout:

        print(
            "\n❌ Request timed out."
        )

        return None


    except requests.exceptions.ConnectionError:

        print(
            "\n❌ Internet connection error."
        )

        return None


    except requests.exceptions.RequestException as error:

        print(
            "\n❌ Request Error:",
            error
        )

        return None


# ============================================================
# GET GITHUB PROFILE
# ============================================================

def get_profile():

    global current_profile
    global current_repositories


    print("\n" + "=" * 70)

    print(
        "                 SEARCH GITHUB PROFILE"
    )

    print("=" * 70)


    username = input(
        "\nEnter GitHub username: "
    ).strip()


    if username == "":

        print(
            "\nUsername cannot be empty."
        )

        return


    profile_url = (
        f"{BASE_URL}/users/{username}"
    )


    print(
        "\nFetching GitHub profile..."
    )


    profile = api_request(
        profile_url
    )


    if profile is None:

        return


    current_profile = profile


    # --------------------------------------------------------
    # GET REPOSITORIES
    # --------------------------------------------------------

    repo_url = (
        f"{BASE_URL}/users/"
        f"{username}/repos?per_page=100"
    )


    print(
        "\nFetching repositories..."
    )


    repositories = api_request(
        repo_url
    )


    if repositories is None:

        repositories = []


    current_repositories = repositories


    # --------------------------------------------------------
    # DISPLAY PROFILE
    # --------------------------------------------------------

    display_profile(
        profile
    )


    # --------------------------------------------------------
    # DISPLAY REPOSITORIES
    # --------------------------------------------------------

    display_repositories(
        repositories
    )


    # --------------------------------------------------------
    # SAVE HISTORY
    # --------------------------------------------------------

    save_history(
        profile,
        repositories
    )


# ============================================================
# DISPLAY PROFILE
# ============================================================

def display_profile(profile):

    print("\n" + "=" * 70)

    print(
        "                 PROFILE INFORMATION"
    )

    print("=" * 70)


    print(
        "\nUsername:",
        profile.get(
            "login",
            "N/A"
        )
    )


    print(
        "Name:",
        profile.get(
            "name",
            "N/A"
        )
    )


    print(
        "Bio:",
        profile.get(
            "bio",
            "No bio available"
        )
    )


    print(
        "Location:",
        profile.get(
            "location",
            "N/A"
        )
    )


    print(
        "Followers:",
        profile.get(
            "followers",
            0
        )
    )


    print(
        "Following:",
        profile.get(
            "following",
            0
        )
    )


    print(
        "Public Repositories:",
        profile.get(
            "public_repos",
            0
        )
    )


    print(
        "Public Gists:",
        profile.get(
            "public_gists",
            0
        )
    )


    print(
        "Account Created:",
        profile.get(
            "created_at",
            "N/A"
        )[:10]
    )


    print(
        "GitHub Profile:",
        profile.get(
            "html_url",
            "N/A"
        )
    )


# ============================================================
# DISPLAY REPOSITORIES
# ============================================================

def display_repositories(repositories):

    print("\n" + "=" * 70)

    print(
        "                 PUBLIC REPOSITORIES"
    )

    print("=" * 70)


    if len(repositories) == 0:

        print(
            "\nNo public repositories found."
        )

        return


    for index, repo in enumerate(
        repositories,
        start=1
    ):

        print(
            f"\n{index}. "
            f"{repo.get('name', 'N/A')}"
        )


        print(
            "   Description:",
            repo.get(
                "description",
                "No description"
            )
        )


        print(
            "   Language:",
            repo.get(
                "language",
                "Not specified"
            )
        )


        print(
            "   ⭐ Stars:",
            repo.get(
                "stargazers_count",
                0
            )
        )


        print(
            "   🍴 Forks:",
            repo.get(
                "forks_count",
                0
            )
        )


        print(
            "   URL:",
            repo.get(
                "html_url",
                "N/A"
            )
        )


# ============================================================
# REPOSITORY STATISTICS
# ============================================================

def repository_statistics():

    print("\n" + "=" * 70)

    print(
        "                 REPOSITORY STATISTICS"
    )

    print("=" * 70)


    if len(current_repositories) == 0:

        print(
            "\nNo repository data available."
        )

        return


    total_repositories = len(
        current_repositories
    )


    total_stars = sum(

        repo.get(
            "stargazers_count",
            0
        )

        for repo
        in current_repositories
    )


    total_forks = sum(

        repo.get(
            "forks_count",
            0
        )

        for repo
        in current_repositories
    )


    print(
        "\nTotal Repositories:",
        total_repositories
    )


    print(
        "Total Stars:",
        total_stars
    )


    print(
        "Total Forks:",
        total_forks
    )


    # --------------------------------------------------------
    # LANGUAGE STATISTICS
    # --------------------------------------------------------

    languages = {}


    for repo in current_repositories:

        language = repo.get(
            "language"
        )


        if language:

            languages[language] = (
                languages.get(
                    language,
                    0
                ) + 1
            )


    print(
        "\nProgramming Languages:"
    )


    if len(languages) == 0:

        print(
            "No language information available."
        )

    else:

        sorted_languages = sorted(
            languages.items(),
            key=lambda x: x[1],
            reverse=True
        )


        for language, count in sorted_languages:

            print(
                f"   {language}: {count} repository(s)"
            )


    # --------------------------------------------------------
    # TOP REPOSITORY
    # --------------------------------------------------------

    top_repository = max(

        current_repositories,

        key=lambda repo:
        repo.get(
            "stargazers_count",
            0
        )
    )


    print(
        "\nMost Starred Repository:"
    )


    print(
        "Name:",
        top_repository.get(
            "name",
            "N/A"
        )
    )


    print(
        "Stars:",
        top_repository.get(
            "stargazers_count",
            0
        )
    )


# ============================================================
# OPEN GITHUB PROFILE
# ============================================================

def open_profile():

    print("\n" + "=" * 70)

    print(
        "                 OPEN GITHUB PROFILE"
    )

    print("=" * 70)


    if current_profile is None:

        print(
            "\nNo profile loaded."
        )

        return


    url = current_profile.get(
        "html_url"
    )


    if url:

        print(
            "\nOpening GitHub profile..."
        )

        webbrowser.open(
            url
        )

    else:

        print(
            "\nProfile URL unavailable."
        )


# ============================================================
# SAVE PROFILE AS FAVORITE
# ============================================================

def save_favorite():

    global favorites


    print("\n" + "=" * 70)

    print(
        "                 SAVE FAVORITE"
    )

    print("=" * 70)


    if current_profile is None:

        print(
            "\nNo profile loaded."
        )

        return


    username = current_profile.get(
        "login",
        ""
    )


    # Check duplicate

    existing_usernames = [

        item["Username"]

        for item in favorites
    ]


    if username in existing_usernames:

        print(
            "\n⭐ Profile already saved."
        )

        return


    record = {

        "Username":
            current_profile.get(
                "login",
                ""
            ),

        "Name":
            current_profile.get(
                "name",
                ""
            ),

        "Followers":
            current_profile.get(
                "followers",
                0
            ),

        "Following":
            current_profile.get(
                "following",
                0
            ),

        "Public Repositories":
            current_profile.get(
                "public_repos",
                0
            ),

        "Location":
            current_profile.get(
                "location",
                ""
            ),

        "Profile URL":
            current_profile.get(
                "html_url",
                ""
            )
    }


    favorites.append(
        record
    )


    print(
        "\n⭐ Profile saved to favorites!"
    )


# ============================================================
# VIEW FAVORITES
# ============================================================

def view_favorites():

    print("\n" + "=" * 70)

    print(
        "                 FAVORITE PROFILES"
    )

    print("=" * 70)


    if len(favorites) == 0:

        print(
            "\nNo favorite profiles."
        )

        return


    for index, profile in enumerate(
        favorites,
        start=1
    ):

        print(
            f"\n{index}. "
            f"{profile['Username']}"
        )


        print(
            "   Name:",
            profile["Name"]
        )


        print(
            "   Followers:",
            profile["Followers"]
        )


        print(
            "   Public Repositories:",
            profile[
                "Public Repositories"
            ]
        )


        print(
            "   URL:",
            profile["Profile URL"]
        )


# ============================================================
# SAVE FAVORITES TO CSV
# ============================================================

def save_favorites():

    if len(favorites) == 0:

        print(
            "\nNo favorites to save."
        )

        return


    try:

        df = pd.DataFrame(
            favorites
        )


        df.to_csv(
            FAVORITES_FILE,
            index=False
        )


        print(
            "\nFavorites saved successfully!"
        )


        print(
            "File:",
            os.path.abspath(
                FAVORITES_FILE
            )
        )


    except Exception as error:

        print(
            "\nError saving favorites:",
            error
        )


# ============================================================
# LOAD FAVORITES
# ============================================================

def load_favorites():

    global favorites


    if not os.path.exists(
        FAVORITES_FILE
    ):

        return


    try:

        df = pd.read_csv(
            FAVORITES_FILE
        )


        favorites = df.to_dict(
            orient="records"
        )


        print(
            f"{len(favorites)} "
            "favorite profile(s) loaded."
        )


    except Exception as error:

        print(
            "\nCould not load favorites:",
            error
        )


# ============================================================
# SAVE SEARCH HISTORY
# ============================================================

def save_history(
    profile,
    repositories
):

    global history


    total_stars = sum(

        repo.get(
            "stargazers_count",
            0
        )

        for repo
        in repositories
    )


    record = {

        "Username":
            profile.get(
                "login",
                ""
            ),

        "Name":
            profile.get(
                "name",
                ""
            ),

        "Followers":
            profile.get(
                "followers",
                0
            ),

        "Following":
            profile.get(
                "following",
                0
            ),

        "Repositories":
            profile.get(
                "public_repos",
                0
            ),

        "Total Stars":
            total_stars,

        "Searched At":
            datetime.now().strftime(
                "%d-%m-%Y %I:%M:%S %p"
            )
    }


    history.append(
        record
    )


# ============================================================
# VIEW HISTORY
# ============================================================

def view_history():

    print("\n" + "=" * 70)

    print(
        "                 SEARCH HISTORY"
    )

    print("=" * 70)


    if len(history) == 0:

        print(
            "\nNo search history available."
        )

        return


    for index, item in enumerate(
        history,
        start=1
    ):

        print(
            f"\n{index}. "
            f"{item['Username']}"
        )


        print(
            "   Followers:",
            item["Followers"]
        )


        print(
            "   Repositories:",
            item["Repositories"]
        )


        print(
            "   Total Stars:",
            item["Total Stars"]
        )


        print(
            "   Searched:",
            item["Searched At"]
        )


# ============================================================
# EXPORT HISTORY
# ============================================================

def export_history():

    if len(history) == 0:

        print(
            "\nNo history available."
        )

        return


    try:

        df = pd.DataFrame(
            history
        )


        df.to_csv(
            HISTORY_FILE,
            index=False
        )


        print(
            "\nHistory exported successfully!"
        )


        print(
            "File:",
            os.path.abspath(
                HISTORY_FILE
            )
        )


    except Exception as error:

        print(
            "\nError exporting history:",
            error
        )


# ============================================================
# PROJECT INFORMATION
# ============================================================

def project_information():

    print("\n" + "=" * 70)

    print(
        "                 PROJECT INFORMATION"
    )

    print("=" * 70)


    print(
        "\nProject Name:"
    )

    print(
        "GitHub Profile Explorer"
    )


    print(
        "\nAPI:"
    )

    print(
        "GitHub REST API"
    )


    print(
        "\nMain Python Libraries:"
    )

    print(
        "• requests"
    )

    print(
        "• pandas"
    )

    print(
        "• os"
    )

    print(
        "• time"
    )

    print(
        "• webbrowser"
    )

    print(
        "• datetime"
    )


    print(
        "\nMain Concepts:"
    )

    print(
        "• REST API"
    )

    print(
        "• JSON"
    )

    print(
        "• API requests"
    )

    print(
        "• Data extraction"
    )

    print(
        "• File handling"
    )

    print(
        "• CSV"
    )

    print(
        "• Data processing"
    )

    print(
        "• Exception handling"
    )


# ============================================================
# MAIN MENU
# ============================================================

def main_menu():

    while True:

        print("\n")

        print("=" * 70)

        print(
            "             🧑‍💻 GITHUB PROFILE EXPLORER"
        )

        print("=" * 70)


        print(
            "1.  Search GitHub Profile"
        )

        print(
            "2.  Repository Statistics"
        )

        print(
            "3.  Open GitHub Profile"
        )

        print(
            "4.  Save Current Profile"
        )

        print(
            "5.  View Favorite Profiles"
        )

        print(
            "6.  Save Favorites to CSV"
        )

        print(
            "7.  View Search History"
        )

        print(
            "8.  Export History to CSV"
        )

        print(
            "9.  Project Information"
        )

        print(
            "10. Exit"
        )


        print("=" * 70)


        choice = input(
            "Enter your choice: "
        ).strip()


        if choice == "1":

            get_profile()


        elif choice == "2":

            repository_statistics()


        elif choice == "3":

            open_profile()


        elif choice == "4":

            save_favorite()


        elif choice == "5":

            view_favorites()


        elif choice == "6":

            save_favorites()


        elif choice == "7":

            view_history()


        elif choice == "8":

            export_history()


        elif choice == "9":

            project_information()


        elif choice == "10":

            print(
                "\n" + "=" * 70
            )

            print(
                "Thank you for using "
                "GitHub Profile Explorer! 🧑‍💻"
            )

            print(
                "Keep learning Python! 🐍"
            )

            print("=" * 70)

            break


        else:

            print(
                "\n❌ Invalid choice!"
            )

            print(
                "Please choose between 1 and 10."
            )


# ============================================================
# START PROJECT
# ============================================================

print("=" * 70)

print(
    "             🧑‍💻 GITHUB PROFILE EXPLORER"
)

print(
    "                  PYTHON MINOR PROJECT"
)

print("=" * 70)


now = datetime.now()


print(
    "Date:",
    now.strftime(
        "%d-%m-%Y"
    )
)


print(
    "Time:",
    now.strftime(
        "%I:%M:%S %p"
    )
)


print("=" * 70)


load_favorites()


print(
    "\n✅ Project started successfully!"
)

print(
    "Choose an option from the menu."
)


main_menu()

             🧑‍💻 GITHUB PROFILE EXPLORER
                  PYTHON MINOR PROJECT
Date: 10-08-2026
Time: 03:49:10 PM

✅ Project started successfully!
Choose an option from the menu.


             🧑‍💻 GITHUB PROFILE EXPLORER
1.  Search GitHub Profile
2.  Repository Statistics
3.  Open GitHub Profile
4.  Save Current Profile
5.  View Favorite Profiles
6.  Save Favorites to CSV
7.  View Search History
8.  Export History to CSV
9.  Project Information
10. Exit
Enter your choice: 1

                 SEARCH GITHUB PROFILE

Enter GitHub username: purnimakushwaha

Fetching GitHub profile...

API Response Time: 0.1817 seconds

Fetching repositories...

API Response Time: 0.1965 seconds

                 PROFILE INFORMATION

Username: purnimakushwaha
Name: Purnima Kushwaha
Bio: None
Location: None
Followers: 0
Following: 0
Public Repositories: 2
Public Gists: 0
Account Created: 2026-07-29
GitHub Profile: https://github.com/purnimakushwaha

                 PUBLIC REPOSITORIES

1. itc101
   Descri